# Machine Learning Training (Tree-Based Models)
This notebook trains 4 tree-based models across 6 commodities:
1. **XGBoost**
2. **LightGBM**
3. **CatBoost**
4. **Random Forest**

It uses strict chronological train/val/test splits, early stopping, and generates live feature importance charts.

In [13]:
import pandas as pd
import numpy as np
import os
import json
import warnings
import joblib
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

data_dir = './data'
results_dir = './results'
models_dir = './models'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
commodities = ['gold', 'silver', 'copper', 'natural_gas', 'crude_oil', 'wheat']


In [14]:
datasets = {}
print("Loading and validating datasets...")
for name in commodities:
    file_path = os.path.join(data_dir, f"{name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
        datasets[name] = df
        print(f"✅ {name.upper()}: {df.shape} | Dates: {df.index.min().date()} to {df.index.max().date()} | NaNs: {df.isna().sum().sum()}")
    else:
        print(f"❌ {name.upper()}: Not found")


Loading and validating datasets...
✅ GOLD: (3174, 36) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0
✅ SILVER: (3174, 35) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0
✅ COPPER: (2914, 34) | Dates: 2015-06-01 to 2026-07-30 | NaNs: 0
✅ NATURAL_GAS: (3183, 36) | Dates: 2014-05-20 to 2026-07-30 | NaNs: 0
✅ CRUDE_OIL: (2587, 35) | Dates: 2016-08-31 to 2026-07-30 | NaNs: 0
✅ WHEAT: (3174, 37) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0


### Train / Validation / Test Split Strategy
- **Train (70%)**: Used to fit the trees.
- **Validation (10%)**: Used for early stopping (preventing overfitting).
- **Test (20%)**: Completely unseen data used for final metrics.


In [15]:
def calculate_metrics(y_true, y_pred, y_true_prev):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAPE handling zeros
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
    
    # R2 Score
    r2 = r2_score(y_true, y_pred)
    
    # Directional Accuracy
    actual_dir = np.sign(y_true - y_true_prev)
    pred_dir = np.sign(y_pred - y_true_prev)
    correct_dir = (actual_dir == pred_dir)
    dir_acc = np.mean(correct_dir) * 100
    
    # Naive Baseline (predict today's price for tomorrow)
    naive_mae = mean_absolute_error(y_true, y_true_prev)
    
    # Improvement vs Naive
    imp_pct = ((naive_mae - mae) / naive_mae) * 100 if naive_mae > 0 else 0
    
    return {
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'MAPE': round(mape, 2),
        'Dir_Acc': round(dir_acc, 2),
        'R2': round(r2, 4),
        'Naive_MAE': round(naive_mae, 4),
        'Improvement_Pct': round(imp_pct, 2)
    }

def get_train_val_test(df):
    drop_cols = ['Target_Close_Next', 'Target_Return_Next', 'Target_Direction', 
                 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    features = [col for col in df.columns if col not in drop_cols]
    
    X = df[features]
    y_return = df['Target_Return_Next']
    
    total_len = len(df)
    train_end = int(total_len * 0.7)
    val_end = int(total_len * 0.8)
    
    X_train = X.iloc[:train_end]
    X_val = X.iloc[train_end:val_end]
    X_test = X.iloc[val_end:]
    
    y_train = y_return.iloc[:train_end]
    y_val = y_return.iloc[train_end:val_end]
    y_test = y_return.iloc[val_end:]
    
    y_true_price_test = df['Target_Close_Next'].iloc[val_end:].values
    y_true_prev_test = df['Close'].iloc[val_end:].values
    
    return X_train, X_val, X_test, y_train, y_val, y_test, y_true_price_test, y_true_prev_test, features


In [16]:
results = {}
feature_importances = {}

for name, df in tqdm(datasets.items(), desc="Commodities"):
    X_train, X_val, X_test, y_train, y_val, y_test, y_true_price, y_true_prev, features = get_train_val_test(df)
    
    eval_set_xgb_lgb = [(X_val, y_val)]
    
    print(f"\n{'='*50}\nTraining {name.upper()}...\n{'='*50}")
    
    res = {}
    fi = {}
    
    # 1. XGBoost
    xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8, 
                       colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42,
                       early_stopping_rounds=20)
    xgb.fit(X_train, y_train, eval_set=eval_set_xgb_lgb, verbose=False)
    xgb_ret = xgb.predict(X_test)
    xgb_price = y_true_prev * (1 + xgb_ret)
    res['XGBoost'] = calculate_metrics(y_true_price, xgb_price, y_true_prev)
    xgb.save_model(os.path.join(models_dir, f'{name}_xgboost_1d.json'))
    fi['XGBoost'] = xgb.feature_importances_
    
    # 2. LightGBM
    lgb = LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8, 
                        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
    # LightGBM newer versions use callbacks for early stopping
    from lightgbm import early_stopping
    lgb.fit(X_train, y_train, eval_set=eval_set_xgb_lgb, callbacks=[early_stopping(stopping_rounds=20, verbose=False)])
    lgb_ret = lgb.predict(X_test)
    lgb_price = y_true_prev * (1 + lgb_ret)
    res['LightGBM'] = calculate_metrics(y_true_price, lgb_price, y_true_prev)
    lgb.booster_.save_model(os.path.join(models_dir, f'{name}_lightgbm_1d.txt'))
    fi['LightGBM'] = lgb.feature_importances_
    
    # 3. CatBoost
    cat = CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, 
                            random_seed=42, verbose=0, early_stopping_rounds=20)
    cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    cat_ret = cat.predict(X_test)
    cat_price = y_true_prev * (1 + cat_ret)
    res['CatBoost'] = calculate_metrics(y_true_price, cat_price, y_true_prev)
    cat.save_model(os.path.join(models_dir, f'{name}_catboost_1d.cbm'))
    fi['CatBoost'] = cat.feature_importances_
    
    # 4. Random Forest
    rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=5, 
                               max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_ret = rf.predict(X_test)
    rf_price = y_true_prev * (1 + rf_ret)
    res['RandomForest'] = calculate_metrics(y_true_price, rf_price, y_true_prev)
    joblib.dump(rf, os.path.join(models_dir, f'{name}_randomforest_1d.pkl'))
    fi['RandomForest'] = rf.feature_importances_
    
    results[name] = res
    feature_importances[name] = pd.DataFrame(fi, index=features)
    
    # Display running results
    df_res = pd.DataFrame(res).T
    display(df_res)


Commodities:   0%|          | 0/6 [00:00<?, ?it/s]


Training GOLD...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,35.1406,56.9665,0.98,53.70,0.9959,35.1844,0.12
LightGBM,35.1265,56.9628,0.97,55.43,0.9959,35.1844,0.16
CatBoost,35.1477,56.9142,0.98,52.76,0.9959,35.1844,0.10
RandomForest,35.5239,56.7235,0.99,46.93,0.9959,35.1844,-0.96



Training SILVER...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,1.0461,2.2801,1.93,48.66,0.9867,1.0461,-0.00
LightGBM,1.0455,2.2872,1.93,53.39,0.9866,1.0461,0.05
CatBoost,1.0441,2.2832,1.93,54.96,0.9866,1.0461,0.19
RandomForest,1.1227,2.2863,2.16,55.43,0.9866,1.0461,-7.33


...

,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,0.1091,0.2164,3.39,48.82,0.9229,0.1091,-0.04
LightGBM,0.1091,0.2145,3.38,48.51,0.9242,0.1091,-0.02
CatBoost,0.1097,0.2147,3.39,48.19,0.9241,0.1091,-0.52
RandomForest,0.1130,0.2133,3.45,48.98,0.9250,0.1091,-3.59



Training CRUDE_OIL...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,25.5239,330.6909,27.87,49.42,-764.1634,1.446,-1665.09
LightGBM,64.1963,224.9803,74.29,49.23,-353.1594,1.446,-4339.45
CatBoost,7.3347,81.3283,8.51,50.00,-45.2800,1.446,-407.23
RandomForest,11.5574,119.0966,13.06,49.61,-98.2450,1.446,-699.25



Training WHEAT...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,7.0041,9.3440,1.24,51.81,0.9536,7.0421,0.54
LightGBM,7.0048,9.3716,1.24,49.92,0.9533,7.0421,0.53
CatBoost,7.0334,9.3560,1.24,48.03,0.9535,7.0421,0.12
RandomForest,7.0213,9.4053,1.24,49.45,0.9530,7.0421,0.30


In [17]:
out_path = os.path.join(results_dir, 'stage2_ml_metrics_v2.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Metrics saved to {out_path}")


Metrics saved to ./results/stage2_ml_metrics_v2.json


In [21]:
# Display feature importances for a selected commodity (e.g., Gold)
for name in commodities:
    df_fi = feature_importances[name]
    # Normalize for comparison
    df_fi_norm = df_fi.div(df_fi.sum(axis=0), axis=1) * 100
    df_fi_norm['Mean'] = df_fi_norm.mean(axis=1)
    df_fi_norm = df_fi_norm.sort_values(by='Mean', ascending=True).tail(15)
    
    fig = px.bar(df_fi_norm, x='Mean', y=df_fi_norm.index, orientation='h', 
                 title=f'{name.upper()}: Top 15 Features Across All 4 Models',
                 labels={'Mean': 'Average Importance (%)', 'index': 'Feature'})
    fig.show()
